# `expansion_hunter_06_progress` — progress, cost, failures

Read-only: checks Batch and the run folder, never submits and never writes the ledger. Run All any time,
including while 05's autopilot is running. Nothing to edit; the run comes from `ACTIVE_RUN`.

- **Estimated cost** is available minutes after jobs finish: each VM is priced from its tasks' start and end
  times plus 85 s of startup, and Nearline at 1.34c per sample that read its CRAM. This model matched the
  Sep 29 bill to within 1%. The Spot price is an assumption (35% of on-demand) until a Spot run is billed.
- **Billed cost** needs `billing_table` set in 03 and read access to the billing export; it lags about a day.

In [ ]:
from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path


def find_repo_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "scripts" / "vwb_batch" / "campaign.py").is_file():
            return p
        nested = p / "aou-lr-phase-2" / "scripts" / "vwb_batch" / "campaign.py"
        if nested.is_file():
            return nested.parents[2]
    raise FileNotFoundError("Cannot find scripts/vwb_batch/campaign.py. git pull kvg/aou-lr-phase-2 and re-run.")


REPO_ROOT = find_repo_root()
sys.path.insert(0, str(REPO_ROOT / "scripts"))
for name in [m for m in sys.modules if m == "vwb_batch" or m.startswith("vwb_batch.")]:
    del sys.modules[name]  # pick up a git pull without restarting the kernel

from vwb_batch import alloc, campaign, cohort, cost, gcs, registry  # noqa: E402
from vwb_batch.pipelines import expansion_hunter as eh  # noqa: E402

OUTPUT_BUCKET_GS = os.environ.get("EH_OUTPUT_BUCKET_GS", "gs://aou-lr-phase2-resources").rstrip("/")
PIPELINE = "expansion_hunter"
RUN_ID = os.environ.get("EH_RUN_ID", "").strip() or campaign.read_active_run(OUTPUT_BUCKET_GS, PIPELINE)
if not RUN_ID:
    raise SystemExit("No active run. Run expansion_hunter_03_setup first (or set EH_RUN_ID).")
PATHS = registry.run_paths(output_bucket=OUTPUT_BUCKET_GS, pipeline=PIPELINE, run_id=RUN_ID)
if shutil.which("gcloud") is None:
    raise SystemExit("gcloud is not on PATH. Open this notebook in a VWB Jupyter app.")
PROJECT, SA = alloc.require_vwb_env()
CODE_VERSION = eh.code_version()
print("RUN_ID:", RUN_ID, "   ledger:", PATHS.prefix)
print("code version:", CODE_VERSION, "   project:", PROJECT)

cfg = campaign.Settings.from_run_json(registry.load_run_json(PATHS))
records = campaign.load_samples(PATHS, cfg)
sample_ids = [r["sample_id"] for r in records]
state = campaign.load_state(PATHS)
campaign.sync(PATHS, state, project=PROJECT, region=cfg.region, save_details=False, log=lambda m: None)
plan = campaign.plan(cfg, sample_ids, state, code_version=CODE_VERSION)
for line in campaign.summary_lines(plan, cfg, total=len(sample_ids)):
    print(line)

## Slide figure

PNG and PDF land in `expansion_hunter_rw_scratch/figures/`.

In [ ]:
from IPython.display import Image, display

from vwb_batch import billing, plots

jobs = list(state.get("jobs", {}).values())
actual_total = None
if cfg.billing_table and jobs:
    starts = sorted(j["submitted_at"] for j in jobs)
    try:
        by_job = billing.vm_cost_by_job(
            table=cfg.billing_table, project=PROJECT, start=starts[0].replace("T", " ").rstrip("Z"),
            end=registry.utcnow().replace("T", " ").rstrip("Z"), job_prefix=f"eh-{alloc.sanitize_label(RUN_ID, max_len=24)}",
        )
        actual_total = sum(by_job.values())
    except Exception as exc:
        print("billing export query failed:", str(exc)[:300])

n = plan.numbers
proj = campaign.projection(plan, cfg)
paths = plots.plot_run_progress(
    Path.cwd() / "expansion_hunter_rw_scratch" / "figures",
    run_id=RUN_ID,
    catalog=cfg.catalog.rsplit("/", 1)[-1],
    counts=n,
    total=len(sample_ids),
    spent_usd=n["spent_usd"],
    committed_usd=n["committed_usd"],
    projected_usd=proj["projected_total_usd"],
    budget_usd=cfg.budget_usd,
    actual_usd=actual_total,
    status_line=plan.explain().split(". Est.")[0],
    updated_at=registry.utcnow(),
)
display(Image(filename=str(paths[0]), width=900))

## Cost by wave

Estimated from Batch task timestamps. `$/done` spreads failed attempts over the samples that finished.

In [ ]:
from collections import defaultdict

rows = defaultdict(lambda: defaultdict(float))
for job in sorted(jobs, key=lambda j: j["submitted_at"]):
    key = (int(job.get("wave") or 0), job.get("kind", ""), job.get("code_version", ""))
    r = rows[key]
    r["jobs"] += 1
    r["samples"] += len(job["sample_ids"])
    ts = job.get("task_states")
    if ts is None:
        r["running"] += len(job["sample_ids"])
        continue
    r["done"] += sum(s == "SUCCEEDED" for s in ts)
    r["failed"] += sum(s not in campaign.NEVER_RAN and s != "SUCCEEDED" for s in ts)
    c = job.get("cost") or {}
    r["vm"] += c.get("vm_usd", 0.0)
    r["nearline"] += c.get("nearline_usd", 0.0)
    r["spot"] += (job.get("provisioning") or "").upper() == "SPOT"
    r["machine"] = job.get("machine") or ""

print(f"{'wave':>4} {'kind':<7} {'code':<10} {'jobs':>4} {'samples':>8} {'done':>7} {'failed':>6} {'running':>7} "
      f"{'VM $':>9} {'Nearline $':>10} {'$/done':>8}  machine")
for (wave, kind, cv), r in rows.items():
    total = r["vm"] + r["nearline"]
    per = f"{total / r['done']:.4f}" if r["done"] else "-"
    spot = " spot" if r["spot"] else ""
    print(f"{wave:>4} {kind:<7} {cv:<10} {int(r['jobs']):>4} {int(r['samples']):>8,} {int(r['done']):>7,} "
          f"{int(r['failed']):>6,} {int(r['running']):>7,} {r['vm']:>9.2f} {r['nearline']:>10.2f} {per:>8}  {r['machine']}{spot}")
s = campaign.spend(state)
print(f"\ntotal estimated: VM ${s['vm_usd']:,.2f} + Nearline ${s['nearline_usd']:,.2f} = ${s['total_usd']:,.2f}")

## Failures

Latest failed attempts, with the logs each task leaves. The tails of the three most recent are printed below.

In [ ]:
failed = campaign.failed_samples(sample_ids, state, max_attempts=cfg.max_attempts, limit=30)
print(f"failed or gave up: {n['failed'] + n['gave_up']:,} (showing {len(failed)})")
for sid, st in failed:
    print(f"  {sid:<12} {st.status:<8} attempts={st.attempts}  job={st.last_job} task={st.last_index}")

for sid, st in failed[:3]:
    print(f"\n===== {sid} ({st.last_job} task {st.last_index})")
    for kind, uri in eh.log_uris(sid, cfg.out_prefix).items():
        if gcs.exists(uri):
            tail = gcs.cat(uri).splitlines()[-15:]
            print(f"--- {kind}: {uri}")
            print("\n".join("    " + line for line in tail))
        else:
            print(f"--- {kind}: (none) {uri}")

## Jobs in flight

In [ ]:
live = [j for j in jobs if j.get("task_states") is None]
if not live:
    print("(no jobs in flight)")
for job in sorted(live, key=lambda j: j["submitted_at"]):
    counts = ", ".join(f"{k.lower()} {v}" for k, v in sorted((job.get("counts") or {}).items()))
    print(f"{job['job_id']:<44} {job['state']:<10} {job['kind']:<7} {len(job['sample_ids']):>5} tasks  {counts}")

## Billed cost (optional)

With `billing_table` set in 03, compares billed VM cost per job with the estimate. Cloud Storage is billed to
the bucket, not the job, so Nearline is shown by SKU for the run's time window.

In [ ]:
if not cfg.billing_table:
    print(billing.CONSOLE_HINT.format(project=PROJECT))
elif jobs:
    start = sorted(j["submitted_at"] for j in jobs)[0].replace("T", " ").rstrip("Z")
    end = registry.utcnow().replace("T", " ").rstrip("Z")
    try:
        by_job = billing.vm_cost_by_job(table=cfg.billing_table, project=PROJECT, start=start, end=end,
                                        job_prefix=f"eh-{alloc.sanitize_label(RUN_ID, max_len=24)}")
        est = {j["job_id"]: (j.get("cost") or {}).get("vm_usd") for j in jobs if j.get("cost")}
        both = [k for k in est if k in by_job]
        print(f"billed VM cost for {len(by_job)} jobs: ${sum(by_job.values()):,.2f}")
        if both:
            e, a = sum(est[k] for k in both), sum(by_job[k] for k in both)
            print(f"estimate vs billed on {len(both)} jobs: ${e:,.2f} vs ${a:,.2f} (ratio {e / a:.3f})")
        for row in billing.storage_cost_by_sku(table=cfg.billing_table, project=PROJECT, start=start, end=end)[:8]:
            print(f"  ${row['usd']:>9,.2f}  {row['sku']}")
    except Exception as exc:
        print("billing export query failed:", str(exc)[:400])